In [10]:
import csv
import base64
import requests
import json

UGS_KEY_ID = "143b38ef-0074-42a9-9d9c-d6611bddb22f"
UGS_SECRET_KEY = "BvGclZ_ETTIfz0Ubh9AVAW3aReoM9nSh"
UGS_PROJECT_ID = "d39f6e74-405c-4aeb-bb11-7d7d195d6cd2"
UGS_ENVIRONMENT_ID = "23ad3682-481d-45de-8acc-d6331193e124"
SCRIPT_NAME = "ExportPlayerStats"

EXPECTED_KEYS = [
    "easy_kills", "easy_damageDealt", "easy_totalMatches", "easy_winRate",
    "medium_kills", "medium_damageDealt", "medium_totalMatches", "medium_winRate",
    "hard_kills", "hard_damageDealt", "hard_totalMatches", "hard_winRate",
    "ai_kills", "ai_damageDealt", "ai_totalMatches", "ai_winRate",
]

def get_stateless_token():
    raw = f"{UGS_KEY_ID}:{UGS_SECRET_KEY}".encode("utf-8")
    basic = base64.b64encode(raw).decode("utf-8")

    token_url = (
        "https://services.api.unity.com/auth/v1/token-exchange"
        f"?projectId={UGS_PROJECT_ID}&environmentId={UGS_ENVIRONMENT_ID}"
    )

    headers = {
        "Authorization": f"Basic {basic}",
        "Accept": "application/json",
    }

    r = requests.post(token_url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json()["accessToken"]

def get_all_player_ids(token):
    # Cloud Save REST Get Players
    url = f"https://cloud-save.services.api.unity.com/v1/data/projects/{UGS_PROJECT_ID}/players"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    }

    player_ids = []
    after = None

    while True:
        params = {}
        if after:
            params["after"] = after

        r = requests.get(url, headers=headers, params=params, timeout=30)
        print("GET PLAYERS STATUS:", r.status_code)
        print("GET PLAYERS BODY:", r.text)
        r.raise_for_status()

        data = r.json()
        results = data.get("results", [])

        for item in results:
            pid = item.get("id")
            if pid:
                player_ids.append(pid)

        # pagination shape can vary by response
        after = data.get("after")
        if not after or not results:
            break

    return player_ids

def get_player_items(token, player_id):
    url = f"https://cloud-save.services.api.unity.com/v1/data/projects/{UGS_PROJECT_ID}/players/{player_id}/items"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    }

    r = requests.get(url, headers=headers, timeout=30)
    print(f"GET ITEMS STATUS [{player_id}]:", r.status_code)
    print(f"GET ITEMS BODY [{player_id}]:", r.text)
    r.raise_for_status()

    return r.json()

def build_rows(player_ids, token):
    rows = []

    for player_id in player_ids:
        payload = get_player_items(token, player_id)
        items = payload.get("results", [])

        row = {"playerId": player_id}

        # default all expected keys to 0
        for key in EXPECTED_KEYS:
            row[key] = 0

        for item in items:
            key = item.get("key")
            if key in EXPECTED_KEYS:
                row[key] = item.get("value", 0)

        rows.append(row)

    return rows

def export_csv(rows, filename="ugs_export.csv"):
    fieldnames = ["playerId"] + EXPECTED_KEYS
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Exported {len(rows)} rows to {filename}")

def main():
    token = get_stateless_token()
    player_ids = get_all_player_ids(token)
    print("PLAYER IDS:", player_ids)

    rows = build_rows(player_ids, token)
    export_csv(rows)

if __name__ == "__main__":
    main()

GET PLAYERS STATUS: 200
GET PLAYERS BODY: {"results":[{"id":"4fhhSDZsRgbSyKyqTCPzTaSz2DuX","accessClasses":{"default":{"numKeys":33,"totalSize":67}}},{"id":"nOGrTOKEzZKl6qurskbx2pzI7hht","accessClasses":{"default":{"numKeys":44,"totalSize":87}}}],"links":{"next":null}}
PLAYER IDS: ['4fhhSDZsRgbSyKyqTCPzTaSz2DuX', 'nOGrTOKEzZKl6qurskbx2pzI7hht']
GET ITEMS STATUS [4fhhSDZsRgbSyKyqTCPzTaSz2DuX]: 200
GET ITEMS BODY [4fhhSDZsRgbSyKyqTCPzTaSz2DuX]: {"results":[{"key":"ai_assists","value":0,"writeLock":"68f59aef5e6a2f8208b35efc5eccd68d","modified":{"date":"2026-03-26T15:23:39Z"},"created":{"date":"2026-03-26T15:12:02Z"}},{"key":"ai_boosts","value":0,"writeLock":"c7c9e54013bcf6cee9bd89fe1b02581a","modified":{"date":"2026-03-26T15:23:39Z"},"created":{"date":"2026-03-26T15:12:02Z"}},{"key":"ai_damageDealt","value":10175,"writeLock":"a7856d62deeedbcee117ce03fbbfdcc2","modified":{"date":"2026-03-26T15:23:39Z"},"created":{"date":"2026-03-26T15:12:02Z"}},{"key":"ai_headshotKills","value":40,"writeLo